In [12]:
"""
PIPELINE STAGE: Final Data Integration, Reshaping & Structural Merging
PERIOD: 2021 - 2024 (Extended Temporal Coverage)
INPUTS: subscriber_2.xlsx, consumption_2.xlsx
OUTPUT: 06_2021_2024_final.xlsx
LIBRARIES: pandas, numpy, os

1. OBJECTIVE:
    Execute the final synchronization of the energy database. This script reshapes long-format 
    subscriber data into wide-format features, aggregates consumption metrics, and performs 
    a precision merge for academic research (PeerJ standards).

2. TEMPORAL ENCODING & NORMALIZATION:
    - Month Mapping: Converts string-based month names (e.g., 'December', 'January') into 
      numerical integers (1-12). Essential for time-series forecasting.
    - Type Enforcement: Forces [Plate, Year, Month] columns into numeric types to eliminate 
      merging errors caused by string-to-number mismatches.

3. SUBSCRIBER DATA RESTRUCTURING (Advanced Pivoting):
    - Block-Wise Processing: Handles structured 5-row repeating blocks containing 
      [Plate, Province, Year, Month].
    - Feature Engineering: Pivots 'Subscriber Type' rows into individual columns: 
      [Lighting, Public, Residential, Industrial, Agricultural].
    - Remainder Filter: Automatically discards incomplete 5-row cycles to maintain matrix alignment.

4. CONSUMPTION DATA AGGREGATION:
    - Dynamic Mapping: Groups data by [Plate, Year, Month] to calculate total regional consumption.
    - Header Standardization: Ensures metrics are stored under the 'Total_Consumption' key.

5. JOINING STRATEGY & FINAL OUTPUT:
    - Left Merge: Uses [Plate, Year, Month] as the composite key to preserve subscriber base integrity.
    - Schema Alignment: Final Column Sequence: [Plate, Province, Year, Month, Total_Consumption, 
      Lighting, Public, Residential, Industrial, Agricultural].
"""

import os
import pandas as pd
import numpy as np

def prepare_data(subscriber_path, consumption_path, output_path):
    # Ay isimlerini sayıya dönüştürmek için kapsamlı sözlük
    MONTH_MAP = {
        "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
        "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
        "ocak": 1, "şubat": 2, "mart": 3, "nisan": 4, "mayıs": 5, "haziran": 6,
        "temmuz": 7, "ağustos": 8, "eylül": 9, "ekim": 10, "kasım": 11, "aralık": 12
    }

    try:
        # 1. ABONE VERİLERİNİ OKU
        print("Abone verileri işleniyor...")
        df_sub = pd.read_excel(subscriber_path)
        df_sub.columns = df_sub.columns.str.strip()

        # VERİ TEMİZLİĞİ: 'Tüketici Sayısı' sayıya çevir
        val_col = 'Tüketici Sayısı'
        df_sub[val_col] = df_sub[val_col].astype(str).str.replace('.', '', regex=False)
        df_sub[val_col] = pd.to_numeric(df_sub[val_col], errors='coerce').fillna(0)

        # 5'in katı kontrolü
        remainder = len(df_sub) % 5
        if remainder != 0:
            df_sub = df_sub.iloc[:-remainder]

        # BLOKLARI SÜTUNLARA DÖNÜŞTÜR
        base_info = df_sub.iloc[::5, :4].reset_index(drop=True)
        base_info.columns = ['Plate', 'Province', 'Year', 'Month']
        
        # --- AY DÖNÜŞÜMÜ (Abone Tablosu) ---
        base_info['Month'] = base_info['Month'].astype(str).str.lower().str.strip().map(MONTH_MAP).fillna(base_info['Month'])
        
        values = df_sub[val_col].values.reshape(-1, 5)
        categories = ['Lighting', 'Public', 'Residential', 'Industrial', 'Agricultural']
        df_categories = pd.DataFrame(values, columns=categories)
        
        df_sub_final = pd.concat([base_info, df_categories], axis=1)

        # 2. TÜKETİM VERİLERİNİ OKU
        print("Tüketim verileri işleniyor...")
        df_cons = pd.read_excel(consumption_path)
        df_cons.columns = df_cons.columns.str.strip()
        
        # --- AY DÖNÜŞÜMÜ (Tüketim Tablosu) ---
        if 'Ay' in df_cons.columns:
            df_cons['Ay'] = df_cons['Ay'].astype(str).str.lower().str.strip().map(MONTH_MAP).fillna(df_cons['Ay'])
        
        target_col = "Tüketim" if "Tüketim" in df_cons.columns else df_cons.columns[-1]
        df_cons_grouped = df_cons.groupby(['Plaka', 'Yil', 'Ay'])[target_col].sum().reset_index()
        df_cons_grouped.columns = ['Plate', 'Year', 'Month', 'Total_Consumption']

        # BİRLEŞTİRME ÖNCESİ TİP EŞİTLEME (Hepsi String yapılıyor ki Merge hatasız olsun)
        for d in [df_sub_final, df_cons_grouped]:
            d['Plate'] = d['Plate'].astype(str).str.strip()
            d['Year'] = d['Year'].astype(str).str.strip()
            d['Month'] = d['Month'].astype(str).str.strip()

        # 3. MERGE
        print("Veriler birleştiriliyor...")
        final_df = pd.merge(df_sub_final, df_cons_grouped, on=['Plate', 'Year', 'Month'], how='left')

        # 4. YENİ KAYIT DESENİ
        output_cols = [
            'Plate', 'Province', 'Year', 'Month', 'Total_Consumption',
            'Lighting', 'Public', 'Residential', 'Industrial', 'Agricultural'
        ]
        
        final_result = final_df[[c for c in output_cols if c in final_df.columns]]

        # 5. KAYDET VE GÖSTER
        final_result.to_excel(output_path, index=False)
        
        print("\n" + "="*40)
        print("BAŞARIYLA TAMAMLANDI")
        print(f"Toplam Satır: {len(final_result)}")
        print("Ay Sütunu Kontrol:", final_result['Month'].unique())
        print("-" * 40)
        print(final_result.head())
        print("="*40)

    except Exception as e:
        print(f"Hata oluştu: {e}")

if __name__ == "__main__":
    SUB_FILE = os.path.join("..", "..", "processed_data", "steps", "05_2021_2024_subscriber_2.xlsx")
    CONS_FILE = os.path.join("..","..", "processed_data", "steps", "05_2021_2024_consumption_2.xlsx")
    OUT_FILE = os.path.join("..", "..", "processed_data", "steps", "06_2021_2024_final.xlsx")

    prepare_data(SUB_FILE, CONS_FILE, OUT_FILE)

Abone verileri işleniyor...
Tüketim verileri işleniyor...
Veriler birleştiriliyor...

BAŞARIYLA TAMAMLANDI
Toplam Satır: 3078
Ay Sütunu Kontrol: ['12' '11' '4' '8' '2' '1' '7' '6' '3' '5' '10' '9']
----------------------------------------
  Plate Province  Year Month  Total_Consumption  Lighting  Public  \
0     1    ADANA  2021    12             752120      6252  171253   
1     1    ADANA  2021    11             568203      6243  170308   
2     1    ADANA  2022     4             595328      6287  173909   
3     1    ADANA  2022     8             846164      6337  174824   
4     1    ADANA  2022    12             624319      6478  176615   

   Residential  Industrial  Agricultural  
0       943695        1850         10121  
1       936587        1829          9529  
2       948707        2309         10245  
3       950709        2307         10002  
4       956449        2328         10146  
